# CIFAR-10 ViT Small 训练 Notebook

这个 notebook 仿照 ResNet 版本，完整串起 CIFAR-10 的导出、索引生成、ViT-small 模型、训练、验证、checkpoint、历史记录和可视化流程。
命令行参数解析使用 `click`，日志使用 `logging`，路径管理使用 `pathlib`，训练曲线可视化使用 `matplotlib`，训练历史 CSV 使用 `pandas`。

默认假设数据位于 `CiFar10/cifar-10-batches-py`，如果你已经有导出的图片目录和索引文件，可以直接跳过数据准备阶段进入训练。

In [10]:
from __future__ import annotations

import logging
import pickle
import random
from pathlib import Path
from typing import Iterable

import click
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from matplotlib.ticker import MaxNLocator
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

DEFAULT_WORKSPACE_DIR = Path.cwd().resolve()
DEFAULT_CIFAR10_DIR = DEFAULT_WORKSPACE_DIR / "CiFar10"
DEFAULT_INPUT_DIR = DEFAULT_CIFAR10_DIR / "cifar-10-batches-py"
DEFAULT_EXPORT_DIR = DEFAULT_INPUT_DIR / "export"
DEFAULT_INDEX_DIR = DEFAULT_CIFAR10_DIR / "dataset_index"
DEFAULT_LOG_DIR = DEFAULT_CIFAR10_DIR / "Log"
DEFAULT_LOG_NAME = "vit_cifar10.log"
DEFAULT_CHECKPOINT_PATH = DEFAULT_LOG_DIR / "vit_cifar10_checkpoint.pth"
DEFAULT_HISTORY_CSV_PATH = DEFAULT_LOG_DIR / "vit_training_history.csv"
DEFAULT_HISTORY_PLOT_PATH = DEFAULT_LOG_DIR / "vit_training_history.png"
DEFAULT_SEED = 42
DEFAULT_EPOCHS = 10
DEFAULT_BATCH_SIZE = 128
DEFAULT_IMAGE_SIZE = 32
DEFAULT_NUM_WORKERS = 0
DEFAULT_LR = 3e-4
DEFAULT_WEIGHT_DECAY = 0.05
DEFAULT_TRAIN_RATIO = 0.9
DEFAULT_PATTERN = "data_batch_*"
DEFAULT_INCLUDE_TEST = False
DEFAULT_USE_AMP = True
DEFAULT_GRAD_CLIP_NORM = 1.0
DEFAULT_MEAN = (0.4914, 0.4822, 0.4465)
DEFAULT_STD = (0.2470, 0.2435, 0.2616)

def set_seed(seed: int = DEFAULT_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def setup_logging(log_dir: Path = DEFAULT_LOG_DIR, log_name: str = DEFAULT_LOG_NAME) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("vit_cifar10")
    logger.setLevel(logging.INFO)
    logger.propagate = False

    for handler in list(logger.handlers):
        logger.removeHandler(handler)
        handler.close()

    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    file_handler = logging.FileHandler(log_dir / log_name, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)
    return logger

## 2. CIFAR-10 批次导出与索引

这一部分先把原始 CIFAR-10 batch 展开为按类别整理的图片目录，再扫描图片目录生成 `train.txt` 和 `val.txt` 索引文件，方便后续按索引驱动数据读取。

In [11]:
def load_meta(meta_path: Path) -> list[str]:
    with meta_path.open("rb") as handle:
        data = pickle.load(handle, encoding="latin1")

    label_names = data.get(b"label_names") or data.get("label_names")
    if label_names is None:
        raise ValueError(f"Missing label_names in {meta_path}")

    return [name.decode("utf-8") if isinstance(name, bytes) else str(name) for name in label_names]

def load_batch(batch_path: Path) -> dict:
    with batch_path.open("rb") as handle:
        return pickle.load(handle, encoding="latin1")

def unpack_batch(batch: dict) -> tuple[object, object, object]:
    images = batch.get(b"data") or batch.get("data")
    labels = batch.get(b"labels") or batch.get("labels")
    filenames = batch.get(b"filenames") or batch.get("filenames")

    if images is None or labels is None or filenames is None:
        raise ValueError("Unexpected CIFAR batch format")

    return images, labels, filenames

def export_batches_by_class(
    input_dir: Path,
    output_dir: Path,
    pattern: str = DEFAULT_PATTERN,
    include_test: bool = False,
    logger: logging.Logger | None = None,
) -> dict[str, int]:
    logger = logger or logging.getLogger("vit_cifar10")
    class_names = load_meta(input_dir / "batches.meta")
    batch_paths = sorted(input_dir.glob(pattern))

    if include_test:
        test_batch = input_dir / "test_batch"
        if test_batch.exists() and test_batch not in batch_paths:
            batch_paths.append(test_batch)

    logger.info(
        "开始导出 CIFAR-10 图片: input_dir=%s output_dir=%s pattern=%s include_test=%s",
        input_dir,
        output_dir,
        pattern,
        include_test,
    )

    counts: dict[str, int] = {}

    for batch_path in batch_paths:
        logger.info("处理批次文件: %s", batch_path.name)
        batch = load_batch(batch_path)
        images, labels, filenames = unpack_batch(batch)

        for flat_image, label, filename in zip(images, labels, filenames):
            class_name = class_names[int(label)]
            class_dir = output_dir / class_name
            class_dir.mkdir(parents=True, exist_ok=True)

            image = flat_image.reshape(3, 32, 32).transpose(1, 2, 0)
            image_name = filename.decode("utf-8") if isinstance(filename, bytes) else str(filename)
            output_path = class_dir / f"{Path(image_name).stem}.png"
            Image.fromarray(image).save(output_path)
            counts[class_name] = counts.get(class_name, 0) + 1

    total = sum(counts.values())
    logger.info("导出完成，共写入 %s 张图片到 %s", total, output_dir)
    for class_name in sorted(counts):
        logger.info("%s: %s", class_name, counts[class_name])

    return counts

def is_image_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def collect_class_folders(root_dir: Path) -> list[Path]:
    return sorted([path for path in root_dir.iterdir() if path.is_dir()])

def collect_samples(root_dir: Path) -> list[tuple[Path, int, str]]:
    class_folders = collect_class_folders(root_dir)
    samples: list[tuple[Path, int, str]] = []

    for class_index, class_folder in enumerate(class_folders):
        for image_path in sorted(class_folder.rglob("*")):
            if is_image_file(image_path):
                relative_path = image_path.relative_to(root_dir).as_posix()
                samples.append((image_path, class_index, relative_path))

    return samples

def split_class_samples(
    samples: list[tuple[Path, int, str]],
    seed: int = DEFAULT_SEED,
    train_ratio: float = DEFAULT_TRAIN_RATIO,
) -> tuple[list[tuple[Path, int, str]], list[tuple[Path, int, str]]]:
    rng = random.Random(seed)
    grouped_samples: dict[int, list[tuple[Path, int, str]]] = {}
    for sample in samples:
        grouped_samples.setdefault(sample[1], []).append(sample)

    train_samples: list[tuple[Path, int, str]] = []
    val_samples: list[tuple[Path, int, str]] = []

    for class_index in sorted(grouped_samples):
        class_samples = grouped_samples[class_index][:]
        rng.shuffle(class_samples)
        total = len(class_samples)
        if total <= 1:
            train_count = total
        else:
            train_count = min(total - 1, max(1, int(total * train_ratio)))

        train_samples.extend(class_samples[:train_count])
        val_samples.extend(class_samples[train_count:])

    return train_samples, val_samples

def write_index_file(index_path: Path, samples: Iterable[tuple[Path, int, str]]) -> int:
    index_path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with index_path.open("w", encoding="utf-8") as handle:
        for _, class_index, relative_path in samples:
            handle.write(f"{relative_path}\t{class_index}\n")
            count += 1
    return count

def build_and_save_indices(
    root_dir: Path = DEFAULT_EXPORT_DIR,
    output_dir: Path = DEFAULT_INDEX_DIR,
    seed: int = DEFAULT_SEED,
    logger: logging.Logger | None = None,
) -> dict[str, int]:
    root_dir = root_dir.resolve()
    output_dir = output_dir.resolve()
    logger = logger or logging.getLogger("vit_cifar10")
    logger.info("开始生成索引文件: root_dir=%s output_dir=%s seed=%s", root_dir, output_dir, seed)
    samples = collect_samples(root_dir)
    logger.info("扫描到图片总数: %s", len(samples))
    train_samples, val_samples = split_class_samples(samples, seed=seed)
    logger.info("划分结果: train=%s val=%s", len(train_samples), len(val_samples))

    test_index = output_dir / "test.txt"
    if test_index.exists():
        test_index.unlink()
        logger.info("已删除旧的 test 索引文件: %s", test_index)

    counts = {
        "train": write_index_file(output_dir / "train.txt", train_samples),
        "val": write_index_file(output_dir / "val.txt", val_samples),
    }
    logger.info("索引文件写入完成: %s", counts)
    return counts

def prepare_data_pipeline(
    input_dir: Path = DEFAULT_INPUT_DIR,
    export_dir: Path = DEFAULT_EXPORT_DIR,
    index_dir: Path = DEFAULT_INDEX_DIR,
    pattern: str = DEFAULT_PATTERN,
    include_test: bool = DEFAULT_INCLUDE_TEST,
    seed: int = DEFAULT_SEED,
    logger: logging.Logger | None = None,
) -> tuple[dict[str, int], dict[str, int]]:
    logger = logger or logging.getLogger("vit_cifar10")
    export_counts = export_batches_by_class(
        input_dir=input_dir,
        output_dir=export_dir,
        pattern=pattern,
        include_test=include_test,
        logger=logger,
    )
    index_counts = build_and_save_indices(
        root_dir=export_dir,
        output_dir=index_dir,
        seed=seed,
        logger=logger,
    )
    return export_counts, index_counts

def build_transforms(image_size: int = DEFAULT_IMAGE_SIZE) -> tuple[transforms.Compose, transforms.Compose]:
    train_transform = transforms.Compose([
        transforms.RandomCrop(image_size, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(DEFAULT_MEAN, DEFAULT_STD),
    ])

    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(DEFAULT_MEAN, DEFAULT_STD),
    ])

    return train_transform, eval_transform

class ImageIndexDataset(Dataset):
    def __init__(self, index_file: Path, root_dir: Path | None = None, transform=None, target_transform=None):
        self.index_file = Path(index_file)
        self.root_dir = Path(root_dir) if root_dir is not None else self.index_file.parent
        self.transform = transform
        self.target_transform = target_transform
        self.samples: list[tuple[Path, int]] = []

        with self.index_file.open("r", encoding="utf-8") as handle:
            for line in handle:
                line = line.strip()
                if not line:
                    continue
                relative_path, label_text = line.split("\t")
                image_path = Path(relative_path)
                if not image_path.is_absolute():
                    image_path = self.root_dir / image_path
                self.samples.append((image_path, int(label_text)))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)
        if self.target_transform is not None:
            label = self.target_transform(label)

        return image, label

def build_dataloaders(
    image_root_dir: Path = DEFAULT_EXPORT_DIR,
    index_dir: Path = DEFAULT_INDEX_DIR,
    batch_size: int = DEFAULT_BATCH_SIZE,
    image_size: int = DEFAULT_IMAGE_SIZE,
    num_workers: int = DEFAULT_NUM_WORKERS,
    logger: logging.Logger | None = None,
) -> tuple[DataLoader, DataLoader, list[str]]:
    index_dir = index_dir.resolve()
    image_root_dir = image_root_dir.resolve()
    train_index = index_dir / "train.txt"
    val_index = index_dir / "val.txt"

    if not train_index.exists() or not val_index.exists():
        raise FileNotFoundError(f"Index files not found in {index_dir}; please generate them first.")

    train_transform, eval_transform = build_transforms(image_size=image_size)
    train_dataset = ImageIndexDataset(train_index, root_dir=image_root_dir, transform=train_transform)
    val_dataset = ImageIndexDataset(val_index, root_dir=image_root_dir, transform=eval_transform)
    class_names = sorted({sample[0].parent.name for sample in train_dataset.samples + val_dataset.samples})

    pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

    logger = logger or logging.getLogger("vit_cifar10")
    logger.info("已加载索引数据集: image_root_dir=%s index_dir=%s", image_root_dir, index_dir)
    logger.info("数据集划分完成: train=%s val=%s", len(train_dataset), len(val_dataset))
    logger.info("类别数量: %s", len(class_names))

    return train_loader, val_loader, class_names

def accuracy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> float:
    predictions = logits.argmax(dim=1)
    return (predictions == targets).float().mean().item()

## 3. ViT Small 模型结构

这一格定义 ViT-small 的 patch embedding、Multi-Head Self-Attention、Transformer block 和分类头。它只负责模型骨架，不展开单独的 forward 路径。

In [12]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size: int = DEFAULT_IMAGE_SIZE, patch_size: int = 4, in_channels: int = 3, embed_dim: int = 384) -> None:
        super().__init__()
        if img_size % patch_size != 0:
            raise ValueError("img_size must be divisible by patch_size")
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) * (img_size // patch_size)
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int, attn_dropout: float = 0.0, proj_dropout: float = 0.0) -> None:
        super().__init__()
        if dim % num_heads != 0:
            raise ValueError("dim must be divisible by num_heads")
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.attn_drop = nn.Dropout(attn_dropout)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, num_tokens, embed_dim = x.shape
        qkv = self.qkv(x).reshape(batch_size, num_tokens, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        query, key, value = qkv[0], qkv[1], qkv[2]

        attention = (query @ key.transpose(-2, -1)) * self.scale
        attention = attention.softmax(dim=-1)
        attention = self.attn_drop(attention)

        x = (attention @ value).transpose(1, 2).reshape(batch_size, num_tokens, embed_dim)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class MLP(nn.Module):
    def __init__(self, dim: int, hidden_dim: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class TransformerBlock(nn.Module):
    def __init__(
        self,
        dim: int,
        num_heads: int,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
        attn_dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, num_heads, attn_dropout=attn_dropout, proj_dropout=dropout)
        self.norm2 = nn.LayerNorm(dim)
        hidden_dim = int(dim * mlp_ratio)
        self.mlp = MLP(dim, hidden_dim, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class ViTSmall(nn.Module):
    def __init__(
        self,
        img_size: int = DEFAULT_IMAGE_SIZE,
        patch_size: int = 4,
        in_channels: int = 3,
        num_classes: int = 10,
        embed_dim: int = 384,
        depth: int = 12,
        num_heads: int = 6,
        mlp_ratio: float = 4.0,
        dropout: float = 0.1,
        attn_dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size, in_channels=in_channels, embed_dim=embed_dim)
        self.num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [
                TransformerBlock(
                    dim=embed_dim,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    dropout=dropout,
                    attn_dropout=attn_dropout,
                )
                for _ in range(depth)
            ]
        )
        self.norm = nn.LayerNorm(embed_dim)
        self.head_drop = nn.Dropout(dropout)
        self.head = nn.Linear(embed_dim, num_classes)

        self._init_weights()

    def _init_weights(self) -> None:
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(x)
        batch_size = x.shape[0]
        cls_token = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return x[:, 0]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return vit_forward(self, x)

def vit_small(
    num_classes: int = 10,
    img_size: int = DEFAULT_IMAGE_SIZE,
    patch_size: int = 4,
    embed_dim: int = 384,
    depth: int = 12,
    num_heads: int = 6,
    mlp_ratio: float = 4.0,
    dropout: float = 0.1,
    attn_dropout: float = 0.0,
) -> ViTSmall:
    return ViTSmall(
        img_size=img_size,
        patch_size=patch_size,
        num_classes=num_classes,
        embed_dim=embed_dim,
        depth=depth,
        num_heads=num_heads,
        mlp_ratio=mlp_ratio,
        dropout=dropout,
        attn_dropout=attn_dropout,
    )

## 4. Forward 流程

这里单独展示 ViT 的前向传播路径：先做 patch embedding，再拼接 class token 和位置编码，经过 Transformer blocks，最后取 class token 做分类。

In [13]:
def vit_forward(model: ViTSmall, x: torch.Tensor) -> torch.Tensor:
    x = model.forward_features(x)
    x = model.head_drop(x)
    logits = model.head(x)
    return logits

## 5. 历史记录与 checkpoint

这一格使用 `pandas` 维护训练历史 CSV，并保存 / 恢复模型、优化器、调度器和 AMP 状态的 checkpoint。这样恢复训练时可以直接接着上一次的曲线继续画。

In [14]:
HISTORY_COLUMNS = [
    "epoch",
    "train_loss",
    "train_acc",
    "val_loss",
    "val_acc",
    "lr",
    "best_val_acc",
]

def empty_history_frame() -> pd.DataFrame:
    return pd.DataFrame(columns=HISTORY_COLUMNS)

def load_training_history_csv(history_csv_path: Path) -> pd.DataFrame:
    history_csv_path = Path(history_csv_path)
    if not history_csv_path.exists():
        return empty_history_frame()

    history_df = pd.read_csv(history_csv_path)
    for column in HISTORY_COLUMNS:
        if column not in history_df.columns:
            history_df[column] = pd.NA
    history_df = history_df[HISTORY_COLUMNS]
    return history_df

def append_training_record(history_df: pd.DataFrame, record: dict[str, float | int]) -> pd.DataFrame:
    record_df = pd.DataFrame([record], columns=HISTORY_COLUMNS)
    if history_df.empty:
        return record_df
    return pd.concat([history_df, record_df], ignore_index=True)

def save_training_history_csv(history_df: pd.DataFrame, history_csv_path: Path) -> None:
    history_csv_path = Path(history_csv_path)
    history_csv_path.parent.mkdir(parents=True, exist_ok=True)
    history_df.to_csv(history_csv_path, index=False)

def plot_training_history(history_csv_path: Path, save_path: Path | None = None, show: bool = True) -> None:
    history_csv_path = Path(history_csv_path)
    if not history_csv_path.exists():
        print("没有可视化的数据，请先完成一次训练。")
        return

    history_df = pd.read_csv(history_csv_path)
    if history_df.empty:
        print("没有可视化的数据，请先完成一次训练。")
        return

    for column in HISTORY_COLUMNS:
        if column in history_df.columns:
            history_df[column] = pd.to_numeric(history_df[column], errors="coerce")

    history_df = history_df.dropna(subset=["epoch"])
    epochs = history_df["epoch"].astype(int).tolist()
    train_loss = history_df["train_loss"].tolist()
    val_loss = history_df["val_loss"].tolist()
    train_acc = history_df["train_acc"].tolist()
    val_acc = history_df["val_acc"].tolist()
    lr = history_df["lr"].tolist()

    figure, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(epochs, train_loss, label="train loss")
    axes[0].plot(epochs, val_loss, label="val loss")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].xaxis.set_major_locator(MaxNLocator(integer=True))

    axes[1].plot(epochs, train_acc, label="train acc")
    axes[1].plot(epochs, val_acc, label="val acc")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].xaxis.set_major_locator(MaxNLocator(integer=True))

    axes[2].plot(epochs, lr, label="lr")
    axes[2].set_title("Learning Rate")
    axes[2].set_xlabel("epoch")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    axes[2].xaxis.set_major_locator(MaxNLocator(integer=True))

    figure.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(save_path, dpi=200, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(figure)

def save_checkpoint(
    checkpoint_path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler | None,
    scaler: GradScaler | None,
    epoch: int,
    best_val_acc: float,
    logger: logging.Logger | None = None,
) -> None:
    logger = logger or logging.getLogger("vit_cifar10")
    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "best_val_acc": best_val_acc,
        },
        checkpoint_path,
)
    logger.info("已保存 checkpoint: %s (best_val_acc=%.4f)", checkpoint_path, best_val_acc)

def load_checkpoint(
    checkpoint_path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler | None,
    scaler: GradScaler | None,
    device: torch.device,
    logger: logging.Logger | None = None,
) -> tuple[int, float]:
    logger = logger or logging.getLogger("vit_cifar10")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    if scaler is not None and checkpoint.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    start_epoch = int(checkpoint.get("epoch", 0)) + 1
    best_val_acc = float(checkpoint.get("best_val_acc", 0.0))
    logger.info("已恢复 checkpoint: %s (start_epoch=%s best_val_acc=%.4f)", checkpoint_path, start_epoch, best_val_acc)
    return start_epoch, best_val_acc

## 6. Train 与 Val 流程

这里放训练一个 epoch 的逻辑和验证逻辑，包含前向、反向、AMP 混合精度、梯度裁剪、参数更新，以及验证集统计。

In [15]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    scaler: GradScaler | None = None,
    use_amp: bool = True,
    max_grad_norm: float = 1.0,
    logger: logging.Logger | None = None,
) -> tuple[float, float]:
    model.train()
    logger = logger or logging.getLogger("vit_cifar10")
    amp_enabled = use_amp and device.type == "cuda"

    running_loss = 0.0
    running_correct = 0.0
    sample_count = 0

    for step, (images, targets) in enumerate(loader, start=1):
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=amp_enabled):
            logits = model(images)
            loss = criterion(logits, targets)

        if amp_enabled and scaler is not None:
            scaler.scale(loss).backward()
            if max_grad_norm > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if max_grad_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        sample_count += batch_size

        if step % 100 == 0:
            logger.info("train step=%s loss=%.4f", step, loss.item())

    average_loss = running_loss / sample_count
    average_acc = running_correct / sample_count
    logger.info("train epoch done: loss=%.4f acc=%.4f", average_loss, average_acc)
    return average_loss, average_acc

@torch.no_grad()
def validate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    use_amp: bool = True,
    logger: logging.Logger | None = None,
) -> tuple[float, float]:
    model.eval()
    logger = logger or logging.getLogger("vit_cifar10")
    amp_enabled = use_amp and device.type == "cuda"

    running_loss = 0.0
    running_correct = 0.0
    sample_count = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        with autocast(enabled=amp_enabled):
            logits = model(images)
            loss = criterion(logits, targets)

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        sample_count += batch_size

    average_loss = running_loss / sample_count
    average_acc = running_correct / sample_count
    logger.info("val epoch done: loss=%.4f acc=%.4f", average_loss, average_acc)
    return average_loss, average_acc

## 7. Click 命令行入口

这一格把参数解析、日志、数据加载、模型初始化、checkpoint 恢复和完整训练循环串起来。需要从终端运行时，可以直接复用这里的 `main`。

In [16]:
@click.command()
@click.option("--input-dir", type=click.Path(path_type=Path, exists=True, file_okay=False, dir_okay=True), default=DEFAULT_INPUT_DIR, show_default=True, help="包含 batches.meta 和 batch 文件的原始 CIFAR-10 目录。")
@click.option("--export-dir", type=click.Path(path_type=Path, file_okay=False, dir_okay=True), default=DEFAULT_EXPORT_DIR, show_default=True, help="导出的图片目录。")
@click.option("--index-dir", type=click.Path(path_type=Path, file_okay=False, dir_okay=True), default=DEFAULT_INDEX_DIR, show_default=True, help="train/val 索引文件输出目录。")
@click.option("--prepare-data/--no-prepare-data", default=False, show_default=True, help="是否重新导出图片并生成索引文件；默认直接复用已有数据。")
@click.option("--epochs", type=int, default=DEFAULT_EPOCHS, show_default=True, help="训练轮数。")
@click.option("--batch-size", type=int, default=DEFAULT_BATCH_SIZE, show_default=True, help="每个 batch 的样本数。")
@click.option("--image-size", type=int, default=DEFAULT_IMAGE_SIZE, show_default=True, help="输入图像尺寸。")
@click.option("--lr", type=float, default=DEFAULT_LR, show_default=True, help="学习率。")
@click.option("--weight-decay", type=float, default=DEFAULT_WEIGHT_DECAY, show_default=True, help="权重衰减。")
@click.option("--num-workers", type=int, default=DEFAULT_NUM_WORKERS, show_default=True, help="DataLoader worker 数量。")
@click.option("--seed", type=int, default=DEFAULT_SEED, show_default=True, help="随机种子。")
@click.option("--use-amp/--no-use-amp", default=DEFAULT_USE_AMP, show_default=True, help="是否启用 AMP 混合精度训练。")
@click.option("--grad-clip-norm", type=float, default=DEFAULT_GRAD_CLIP_NORM, show_default=True, help="梯度裁剪阈值，0 表示不裁剪。")
@click.option("--log-dir", type=click.Path(path_type=Path, file_okay=False, dir_okay=True), default=DEFAULT_LOG_DIR, show_default=True, help="日志输出目录。")
@click.option("--history-csv-path", type=click.Path(path_type=Path, file_okay=True, dir_okay=False), default=DEFAULT_HISTORY_CSV_PATH, show_default=True, help="训练结果 CSV 文件路径。")
@click.option("--checkpoint-path", type=click.Path(path_type=Path, file_okay=True, dir_okay=False), default=DEFAULT_CHECKPOINT_PATH, show_default=True, help="训练 checkpoint 保存路径。")
@click.option("--resume-path", type=click.Path(path_type=Path, file_okay=True, dir_okay=False), default=None, help="恢复训练的 checkpoint 路径。")
def main(
    input_dir: Path,
    export_dir: Path,
    index_dir: Path,
    prepare_data: bool,
    epochs: int,
    batch_size: int,
    image_size: int,
    lr: float,
    weight_decay: float,
    num_workers: int,
    seed: int,
    use_amp: bool,
    grad_clip_norm: float,
    log_dir: Path,
    history_csv_path: Path,
    checkpoint_path: Path,
    resume_path: Path | None,
 ) -> pd.DataFrame:
    set_seed(seed)
    logger = setup_logging(log_dir=log_dir)
    device = get_device()
    amp_enabled = use_amp and device.type == "cuda"
    logger.info("使用设备: %s", device)
    logger.info("AMP enabled: %s", amp_enabled)
    logger.info("Gradient clip norm: %.4f", grad_clip_norm)

    history_csv_path = Path(history_csv_path)
    if resume_path is None and history_csv_path.exists():
        history_csv_path.unlink()
        logger.info("已清理旧的训练结果 CSV: %s", history_csv_path)

    if prepare_data:
        export_counts, index_counts = prepare_data_pipeline(
            input_dir=input_dir,
            export_dir=export_dir,
            index_dir=index_dir,
            pattern=DEFAULT_PATTERN,
            include_test=DEFAULT_INCLUDE_TEST,
            seed=seed,
            logger=logger,
        )
        logger.info("导出统计: %s", export_counts)
        logger.info("索引统计: %s", index_counts)
    else:
        train_index = index_dir / "train.txt"
        val_index = index_dir / "val.txt"
        if not export_dir.exists() or not train_index.exists() or not val_index.exists():
            raise FileNotFoundError(
                f"未找到已准备好的数据: export_dir={export_dir}, train_index={train_index}, val_index={val_index}. 如果这是首次运行，请把 prepare_data 打开。"
            )
        logger.info("跳过数据导出和索引生成，直接复用已有数据。")

    train_loader, val_loader, class_names = build_dataloaders(
        image_root_dir=export_dir,
        index_dir=index_dir,
        batch_size=batch_size,
        image_size=image_size,
        num_workers=num_workers,
        logger=logger,
    )

    model = vit_small(num_classes=len(class_names), img_size=image_size).to(device)
    criterion = nn.CrossEntropyLoss()

    decay_parameters: list[nn.Parameter] = []
    no_decay_parameters: list[nn.Parameter] = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.endswith("bias") or "norm" in name.lower() or "pos_embed" in name or "cls_token" in name:
            no_decay_parameters.append(param)
        else:
            decay_parameters.append(param)

    optimizer = torch.optim.AdamW(
        [
            {"params": decay_parameters, "weight_decay": weight_decay},
            {"params": no_decay_parameters, "weight_decay": 0.0},
        ],
        lr=lr,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    scaler = GradScaler(enabled=amp_enabled)

    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    best_val_acc = 0.0
    start_epoch = 1
    history_df = load_training_history_csv(history_csv_path) if resume_path is not None else empty_history_frame()

    if resume_path is not None:
        resume_path = Path(resume_path)
        if not resume_path.exists():
            raise FileNotFoundError(f"Checkpoint not found: {resume_path}")
        start_epoch, best_val_acc = load_checkpoint(
            checkpoint_path=resume_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler if amp_enabled else None,
            device=device,
            logger=logger,
        )
        if history_csv_path.exists():
            logger.info("已载入历史 CSV: %s", history_csv_path)
        else:
            logger.info("未找到历史 CSV，将从当前训练开始写入: %s", history_csv_path)

    for epoch in range(start_epoch, epochs + 1):
        logger.info("===== Epoch %s/%s =====", epoch, epochs)
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
            scaler=scaler if amp_enabled else None,
            use_amp=amp_enabled,
            max_grad_norm=grad_clip_norm,
            logger=logger,
        )
        val_loss, val_acc = validate(
            model,
            val_loader,
            criterion,
            device,
            use_amp=amp_enabled,
            logger=logger,
        )
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_checkpoint(
                checkpoint_path=checkpoint_path,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler if amp_enabled else None,
                epoch=epoch,
                best_val_acc=best_val_acc,
                logger=logger,
            )
            logger.info("本轮验证集指标刷新，已保存最优权重。")

        history_df = append_training_record(
            history_df,
            {
                "epoch": epoch,
                "train_loss": float(train_loss),
                "train_acc": float(train_acc),
                "val_loss": float(val_loss),
                "val_acc": float(val_acc),
                "lr": float(current_lr),
                "best_val_acc": float(best_val_acc),
            },
        )
        save_training_history_csv(history_df, history_csv_path)

        epoch_checkpoint_path = checkpoint_path.parent / f"checkpoint_{epoch}.pth"
        save_checkpoint(
            checkpoint_path=epoch_checkpoint_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler if amp_enabled else None,
            epoch=epoch,
            best_val_acc=best_val_acc,
            logger=logger,
        )

        plot_training_history(history_csv_path, save_path=Path(log_dir) / "vit_training_history.png", show=False)
        logger.info(
            "epoch=%s train_loss=%.4f train_acc=%.4f val_loss=%.4f val_acc=%.4f lr=%.6f best_val_acc=%.4f",
            epoch,
            train_loss,
            train_acc,
            val_loss,
            val_acc,
            current_lr,
            best_val_acc,
        )

    logger.info("训练结束，最佳验证准确率: %.4f", best_val_acc)
    return history_df

## 8. Notebook 训练入口

这里直接调用 `main.callback(...)`，在 notebook 里启动一次完整训练。默认先跑 1 个 epoch，确认数据加载、训练、验证、checkpoint 和曲线保存都能串起来。

In [ ]:
# notebook 训练入口：默认先跑 1 个 epoch，确认训练流程完整可用。
history_df = main.callback(
    input_dir=DEFAULT_INPUT_DIR,
    export_dir=DEFAULT_EXPORT_DIR,
    index_dir=DEFAULT_INDEX_DIR,
    prepare_data=False,
    epochs=1,
    batch_size=32,
    image_size=DEFAULT_IMAGE_SIZE,
    lr=DEFAULT_LR,
    weight_decay=DEFAULT_WEIGHT_DECAY,
    num_workers=DEFAULT_NUM_WORKERS,
    seed=DEFAULT_SEED,
    use_amp=DEFAULT_USE_AMP,
    grad_clip_norm=DEFAULT_GRAD_CLIP_NORM,
    log_dir=DEFAULT_LOG_DIR,
    history_csv_path=DEFAULT_HISTORY_CSV_PATH,
    checkpoint_path=DEFAULT_CHECKPOINT_PATH,
    resume_path=None,
 )
plot_training_history(DEFAULT_HISTORY_CSV_PATH, save_path=DEFAULT_HISTORY_PLOT_PATH)

2026-05-27 15:24:44,269 INFO 使用设备: cpu
2026-05-27 15:24:44,273 INFO AMP enabled: False
2026-05-27 15:24:44,273 INFO Gradient clip norm: 1.0000
2026-05-27 15:24:44,273 INFO 跳过数据导出和索引生成，直接复用已有数据。
2026-05-27 15:24:44,623 INFO 已加载索引数据集: image_root_dir=D:\Jupyter Lab\CiFar10\cifar-10-batches-py\export index_dir=D:\Jupyter Lab\CiFar10\dataset_index
2026-05-27 15:24:44,623 INFO 数据集划分完成: train=45000 val=5000
2026-05-27 15:24:44,623 INFO 类别数量: 10
C:\Users\34354\AppData\Local\Temp\ipykernel_8388\3817307886.py:102: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=amp_enabled)
2026-05-27 15:24:44,771 INFO ===== Epoch 1/1 =====
C:\Users\34354\AppData\Local\Temp\ipykernel_8388\1500844554.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp_enabled):


## 9. 前向测试

这个单元只是一个轻量 smoke test，用随机输入检查 ViT-small 的输出维度是否正常，方便快速确认前向路径没有写错。

In [17]:
device = get_device()
set_seed(DEFAULT_SEED)
model = vit_small(num_classes=10, img_size=DEFAULT_IMAGE_SIZE).to(device)
dummy_input = torch.randn(2, 3, DEFAULT_IMAGE_SIZE, DEFAULT_IMAGE_SIZE, device=device)
dummy_output = model(dummy_input)
print("Smoke test output shape:", tuple(dummy_output.shape))

Smoke test output shape: (2, 10)
